# Lecture 7:  OO Plotting and Subplots

Due to the snow storm, there will be no in-class lecture today, instead please run this already-filled out notebook, and try to understand the answers in terms of the dialog in the Markdown cells.

**Note some cells have commented expressions, these are meant to be enabled and for you to see what effect they have.**

Although a lot of this is a rehearsal what we have done before, some of this may be especially useful for your homework that is due on 2/13.
 

## Today: 2/12 (snow day)

1. Vectorized math
2. Dot products and cross products of vectors
3. The fig and ax objects for OO plotting
4. Adding grids; logarithmic scales, axis tick marks and labels.
5. Multiple subplots.
6. Latex; greek letters, superscripts, and subscripts

 References in Hill's book:
 ```
  Sec. 6.1 (Basic Array Methods), except 6.1.2 (dtypes),
       6.1.7 (broadcasting), 6.1.9 (sorting an array),
       6.1.10 (structured arrays)
  Sec. 2.4.2 (tuples)
  Sec. 7.1 (Line plots and scatter plots)
  Sec. 7.2 (Plot Customization and Refinement)
```

In [ ]:
#  as usual, we will need to import some modules and we give them short aliases

import numpy as np
import matplotlib.pyplot as plt


## Vectorizable or not?

Let us first define a (small) vector `xx` that we can use in vectorized
expressions:

In [ ]:
xx = np.linspace(-2,2,11)
print(xx)

# and a vectorizing expression where we don't need to loop over all elements

yy = xx**2 + 1
print(yy)

We take this last expression and turn it into a function `f`

In [ ]:
def f(x):
    """ a vectorizable function, the argument can be anything that supports the ** operator, viz.
        integer, float, complex or ndarray (and maybe more)
    """
    return x**2+1

so we can in one line of code take the function at each element of the ``xx`` vector:

In [ ]:
yy = f(xx)
print(yy)

Here is a function `g` that is not vectorizable, which we can do to add for example an if-statement

In [ ]:
def g(x):
    """ not a vectorizable function because of the if-statement
    """
    if x < 0:          # this implies it cannot be an array
        return x**3    # as it would compute to a boolean array
    else:
        return x**2
    print("note it should never get here")
    return np.nan

In [ ]:
zz = g(xx)

this will not work "as is", the failure here was intended

A (slow) manual looping for the array is needed instead:

In [ ]:
zz = np.zeros(len(xx))    # we need a placeholder first
for i in range(len(xx)):
    zz[i] = g(xx[i])      # fill the elements, in a loop
print(zz)

but this particular function `g` is simple enough that there is a construct in numpy that emulates the if-statement. This `np.where()` function can still do an element-by-element operation.

Here is that function rewritten with `np.where`:

In [ ]:
def h(x):
    """ this is now a vectorizable function
    """
    return np.where(x<0,x**3,x**2)

now we can do the fast one-liner vectorized operation again


In [ ]:
zz = h(xx)
print(zz)

How can we avoid having to loop when a function cannot be simplified by vectorizable numpy functions?

enter `np.vectorize`.

From the documentation: The `vectorize` function is provided primarily for convenience, not for performance. The implementation is essentially a for loop.

In [ ]:
vg = np.vectorize(g)       # turn 'g' into a vectorizable function

zz = vg(xx)
print(zz)

## Performance

A small diversion on performance.  We write a helper function `loop1` that takes a function and vector and
computes either a vectorized or manual looping resulting vector

In [ ]:
def loop1(func,x,vectorize=False):
    """ helper function to compare and measure
    vectorized vs. looping performance
    """
    if vectorize:
        return func(x)
    else:
        y = np.zeros(len(x))
        for i in range(len(x)):
            y[i] = func(x[i])
        return y
    

If we take our previous small sample `xx` array of 11 elements, the measurement will not be very fair, a longer vector is needed. 

A computer scientist might also be interested in that performance increase as function of the arraysize. 

In [ ]:
nbig = 100001
#nbig = 11
xx = np.linspace(-2,2,nbig)
y1 = f(xx)
y2 = h(xx)

Now that the array has a long length, we could time the performance difference between vectorized and looping operations:   

You should probably see quite a huge difference between them

In [ ]:
%timeit z1 = loop1(f,xx,True)
%timeit z2 = f(xx)
%timeit z3 = loop1(f,xx,False)
%timeit z4 = loop1(g,xx,False)
%timeit z5 = loop1(h,xx,True)
%timeit z6 = vg(xx)


## Arrays: Views vs. Copies

Lets review **views** and **copies** of arrays again. python lists and numpy arrays really act differently in this sense!

Similar confusion can happen with the `unravel` and `flatten`, the latter returns a view and can be more efficient since it does not make a copy, whereas `unravel` makes a copy.

In [ ]:
a = np.arange(6)        # 1-dimensional array
b = a.reshape(3,2)      # view in 3 rows and 2 columns, this is not a copy
c = a[::2]              # every odd element of the array
d = a.copy()            # copy 
e = b.reshape(2,3)      # reshape, can do either a or b
a[0] = 99               # modify the first element
a.resize(2,3)           # new shape for a

print(a)
print(b)
print(c)
print(d)
print(e)

Now look carefully when we set "the first element" of the 'a' array to -99.

Do you understand the output?

Different things will happpen depending on where you say `a[0] = 99` in the sequence. We can clarify this by setting `a[0] = -99` now

In [ ]:
a[0] = -99

print(a)
print(b)
print(c)
print(d)
print(e)

## Some more plotting, OO style

Execute this graph, and study the things you see. You can also study this by commenting out a statement and seeing what effect it has. For example, enable the `set_yscale`, or comment out the `set_ylim` statement.

In [ ]:
fig,ax = plt.subplots()

# we saw earlier already that ax.plot() returns an object
line1, = ax.plot(xx,y1,label='y1')
line2, = ax.plot(xx,y2,marker='v',label='y2')
# it allows to set a dash style manually
line1.set_dashes([2,4,8,4,2,4]) 

ax.set_xlim(-3,3)
ax.set_ylim(bottom=-5)
ax.set_xlabel('X-axis')
ax.set_title('my plot')

y3 = y1 - y2
colors = list(range(len(xx)))   # listing 7.1 in Hill is wrong
ax.scatter(xx,y3,c=colors, s=(xx+2)*40,label='y3=y1-y2')
ax.grid(True)
ax.set_xscale('linear')
#ax.set_yscale('asinh')    # if you mis-spel, it will remind you what's possible
ax.legend(loc="lower right")

plt.tight_layout()


In this figure you see this color changing curve, This is because the `c=` parameter in the `ax.scatter` is an array matching the size of the input, so each point has a different color (and size in fact). Because there are so many points, they merge in a colorful band that changes size.  If you set `nbig=21` or so (a few cells back), it will be


Now a figure with two subplots, left and right, so 1 row, and 2 columns

In [ ]:
fig = plt.figure()
ax1 = fig.add_subplot(1,2,1)   # one row, 2 columns, first plot
ax2 = fig.add_subplot(1,2,2)   #                     second plot
ax1.plot(xx,y1)
ax2.plot(xx,y2)
ax1.set_ylim(-4,4)
ax2.set_ylim(-4,4)
#plt.savefig('fig4.png')

# use some latex, and sub and super scripts.  Use the raw-string version
ax2.text(0,0,r'The Origin $\alpha \dot x_{12} + x^2$')
ax2.scatter([0],[0])
ax2.grid(True) 

#ax2.clear()
plt.tight_layout()

Now create a whole grid of sub-plots, 2 rows and 3 columns.   Too much work to use manual ax1, ax2, ... object, so we create a 2-dim array of `ax` objects!

In [ ]:
fig = plt.figure()
ax = fig.subplots(2,3)    # ax is now an ndarray of Axis objects

print(type(ax))
print(type(ax[0,0]))

# pick only two in this grid to plot 
ax[1,1].plot(xx,y1)
ax[0,2].plot(xx,y2)

# clean up
#plt.tight_layout()

The first two lines in the previous cell can also be combined:

In [ ]:
#     a single plt.subplots() can also be used  (compare previous cell)
fig,ax = plt.subplots(2,3)
#fig.tight_layout()                  # notice the overlap if you don't tight_layout !!

and finally....

In [ ]:
fig = plt.figure()
# nrows=1
# ncols=2
ax1 = fig.add_subplot(1,2,1)    # using three arguments
ax2 = fig.add_subplot(122)      # each integer is an argument, so like 1,2,2
#ax3 = fig.add_subplot(9,9,4)
line1, = ax2.plot(xx,y2)
line2, = ax1.plot(xx,y1)
ax1.set_ylim(-2,2)
ax2.set_ylim(-2,2)
line1.set_dashes([1,4,1])
ax1.grid(True)
ax1.scatter([0],[0],s=100,color='red')
ax1.set_xlabel('X axis')
ax1.set_title('fig1a')
ax2.set_title('fig1b')

colors=list(range(11))          # this would fail if 'xx' is not 11 in length
colors=list(range(len(xx)))
ax2.scatter(xx,y1,s=(xx+2)*40,c=colors)
fig.suptitle("My two figures")

## Steps to play around with

Most of these we have now seen!

1. Plot a function; then plot a portion of the function with slicing.
2. Make use of numpy’s built-in functions for vectorization.
3. Evaluate a user-defined function using vectorization.
4. Protect against divide by zero errors with an if statement and np.nan.
5. Vector dot products and cross products.
6. Get the fig and ax objects with plt.figure() and plt.subplots().
7. Create a plot with the ax object. Set its x and y limits.
8. Add a grid to the plot.
9. Change the x-axis to log scale and back to linear.
10. Add axis labels and a plot title.
11. Add a legend.
12. Set the tick marks.
13. Create multiple subplots.
14. Make a plot in each subplot.
15. Clear a subplot and plot it again.
16. Adjust the figure size.
17. Add text to the plot.
18. Use greek letters, superscripts, and subscripts.

## put it all together! (and practice plotting)

In [ ]:
# let's define a non-vectorizable function
def myfunc(x):
    if np.isclose(x,0):
        # dodge the singularity
        return 0.0
    else:
        return 1/x 

# and vectorize it (remember it's slow but just for convenience)
vfunc = np.vectorize(myfunc)

# another option is to use "np.where(np.isclose(x,0),0,1/x)"
# then we don't need vfunc(), but can use myfunc()



# x-axis range
x = np.linspace(-3,3,20)      # this avoids a zero in the array
#x = np.linspace(-3,3,20)    # try this one, it includes a zero in the array
y = vfunc(x)

fig, ax = plt.subplots()
ax.plot(x,y,"o", label="my special function")
ax.set_xlabel("x")
ax.set_ylabel("y")

# also, let's just plot a range
is_gtr_zero = x > 0
ax.plot(x[is_gtr_zero], y[is_gtr_zero], 'rx', label="Points with x>0")    # overplot
ax.legend()